In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import joblib
data = joblib.load('../data/processed_data.pkl')
df = data['df']
X_train_processed = data['X_train_processed']
y_test = data['y_test']
X_test_processed = data['X_test_processed']
best_rf = joblib.load('../data/best_model.pkl')

# 4. Interpretacja modelu i wnioski biznesowe

Na podstawie wykresu ważności cech widać, że największy wpływ na predykcję mają:
- lead_time
- avg_price_per_room
- no_of_special_requests
- arrival_date, arrival_month
- market_segment_type
- no_of_wkke_nights, no_of_weekend_nights.

Częściowo pokrywa się to z stawianymi hipotezami. Ustaliliśmy już, że im wcześniej robiono rezerwację, tym większa szansa na anulowanie. Jest to najważniejszy czynnik, dlatego dobrze jest się mu przyjrzeć.

### Lead_time

In [ ]:
target_col = 'booking_status'

df_plot = df.copy()

df_plot['lead_time_bin'] = pd.cut(
    df_plot['lead_time'],
    bins=[-1, 7, 30, 90, 180, 365, df_plot['lead_time'].max()],
    labels=['0-7', '8-30', '31-90', '91-180', '181-365', '365+']
)

lead_cancel_rate = df_plot.groupby('lead_time_bin')[target_col].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=lead_cancel_rate, x='lead_time_bin', y=target_col)
plt.title('Wskaźnik anulacji w zależności od lead_time')
plt.xlabel('Lead time (dni)')
plt.ylabel('Średni wskaźnik anulacji')
plt.tight_layout()
plt.show()

In [ ]:
df_plot['lead_time_bin']

### Avg_price_per_room

To druga najważniejszza cecha. Zazwyczaj wyższa cena zwiększa ryzyko anulacji - klienci mogą szukać tańszych zamienników.

In [ ]:
df_plot = df.copy()

df_plot['price_bin'] = pd.qcut(df_plot['avg_price_per_room'], q=10, duplicates='drop')

price_cancel_rate = df_plot.groupby('price_bin')[target_col].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(data=price_cancel_rate, x='price_bin', y=target_col)
plt.title('Wskaźnik anulacji vs poziom ceny pokoju')
plt.xlabel('Przedziały ceny (decyle)')
plt.ylabel('Średni wskaźnik anulacji')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### No_of_special_requests

Zazwyczaj klienci, którzy zgłaszają specjalne życzenia są bardziej zdecydowani i mniej skłonni do anulowania.

In [ ]:
special_req_cancel = df.groupby('no_of_special_requests')[target_col].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(data=special_req_cancel, x='no_of_special_requests', y=target_col)
plt.title('Wskaźnik anulacji vs liczba specjalnych życzeń')
plt.xlabel('Liczba specjalnych życzeń')
plt.ylabel('Średni wskaźnik anulacji')
plt.tight_layout()
plt.show()

### Arrival_month, arrival_date
Moment przyjazdu wpływa na przewidywanie anulacji, najpewniej przez sezonowość pobytu, święta/ długie weekendy. Widać, że najwięcej anulowanych rezerwacji przypada na miesiące letnie, kiedy to plany klientów mogą być bardziej podatne na zmiany.

In [ ]:
month_cancel = df.groupby('arrival_month')[target_col].mean().reset_index()

month_cancel = month_cancel.sort_values('arrival_month')

plt.figure(figsize=(10, 5))
sns.lineplot(data=month_cancel, x='arrival_month', y=target_col, marker='o')
plt.title('Wskaźnik anulacji w zależności od miesiąca przyjazdu')
plt.xlabel('Miesiąc przyjazdu')
plt.ylabel('Średni wskaźnik anulacji')
plt.tight_layout()
plt.show()

### Market_segment_type
Różne segmenty zachowują sie inaczej względem anulacji.

In [ ]:
segment_cancel = (
    df.groupby('market_segment_type')[target_col]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 6))
sns.barplot(data=segment_cancel, x=target_col, y='market_segment_type')
plt.title('Wskaźnik anulacji wg segmentu klienta')
plt.xlabel('Średni wskaźnik anulacji')
plt.ylabel('Segment klienta')
plt.tight_layout()
plt.show()

### No_of_week_nights + no_of_weekend_nights

Wykres sugeruje wzrost anulacji dla dłuższych pobytów, jednak boxploty rozkłądu cech pokazują, że długie pobyty występują relatywnie rzadko. Najbardziej wiarygodna interpretacja dotyczy krótkich i średnich pobytów, które dominują w zbiorze danych.

In [ ]:
df_plot = df.copy()
df_plot['total_nights'] = df_plot['no_of_week_nights'] + df_plot['no_of_weekend_nights']

nights_cancel = df_plot.groupby('total_nights')[target_col].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=nights_cancel, x='total_nights', y=target_col)
plt.title('Wskaźnik anulacji vs długość pobytu')
plt.xlabel('Łączna liczba nocy')
plt.ylabel('Średni wskaźnik anulacji')
plt.tight_layout()
plt.show()

Sprawdźmy, czy cena pokoju łączy się też z segmentem klienta.

In [ ]:

df_plot = df.copy()

df_plot['price_bin'] = pd.qcut(df_plot['avg_price_per_room'], q=10, duplicates='drop')

segment_price_cancel = (
    df_plot.groupby(['price_bin', 'market_segment_type'])[target_col]
    .mean()
    .reset_index()
)
segment_price_cancel['price_bin'] = segment_price_cancel['price_bin'].astype(str)

plt.figure(figsize=(14, 6))
sns.lineplot(
    data=segment_price_cancel,
    x='price_bin',
    y=target_col,
    hue='market_segment_type',
    marker='o'
)

plt.title('Wskaźnik anulacji vs cena pokoju (wszystkie segmenty)')
plt.xlabel('Przedziały ceny (decyle)')
plt.ylabel('Średni wskaźnik anulacji')
plt.xticks(rotation=45)
plt.legend(title='Segment klienta', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

Oprócz tego, że klienci biznesowi są mniej skłonni do anulacji, widać, że wpływ ceny nie jest taki sam w każdym segmencie. 
- w Offline i Online ryzyko anulacji jest bardziej związane z ceną
- w Corporate efekt jest trochę słabszy, jednak nadal związany z ceną
- w małych segmentach (Aviation) dane są niestabilne, a w Complementary dla wszystkich cen pokojów mamy 0 anulacji

Sprawdźmy, czy wpływ lead_time na anulacje różni się między segmentami.


In [ ]:

df_plot = df.copy()

df_plot['lead_time_bin'] = pd.cut(
    df_plot['lead_time'],
    bins=[-1, 7, 30, 90, 180, 365, df_plot['lead_time'].max()],
    labels=['0-7', '8-30', '31-90', '91-180', '181-365', '365+']
)
lead_segment_cancel = (
    df_plot.groupby(['lead_time_bin', 'market_segment_type'])[target_col]
    .mean()
    .reset_index()
)
lead_segment_count = (
    df_plot.groupby(['lead_time_bin', 'market_segment_type'])
    .size()
    .reset_index(name='n')
)

lead_segment_summary = lead_segment_cancel.merge(
    lead_segment_count, on=['lead_time_bin', 'market_segment_type']
)

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=lead_segment_summary,
    x='lead_time_bin',
    y=target_col,
    hue='market_segment_type',
    marker='o'
)

plt.title('Wskaźnik anulacji vs lead time (wg segmentu klienta)')
plt.xlabel('Lead time (dni, przedziały)')
plt.ylabel('Średni wskaźnik anulacji')
plt.legend(title='Segment klienta', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

Czy liczba special request obniża anulacje tak samo w każdym segmencie?

In [ ]:
target = 'booking_status'

df_plot = df.copy()
df_plot = df_plot[df_plot['no_of_special_requests'].isin([0, 1, 2])].copy()

req_segment_cancel = (
    df_plot.groupby(['no_of_special_requests', 'market_segment_type'])[target]
    .mean()
    .reset_index()
)

req_segment_count = (
    df_plot.groupby(['no_of_special_requests', 'market_segment_type'])
    .size()
    .reset_index(name='n')
)

req_segment_summary = req_segment_cancel.merge(
    req_segment_count, on=['no_of_special_requests', 'market_segment_type']
)

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=req_segment_summary,
    x='no_of_special_requests',
    y=target,
    hue='market_segment_type',
    marker='o'
)

plt.title('Wskaźnik anulacji vs liczba specjalnych życzeń (wg segmentu)')
plt.xlabel('Liczba specjalnych życzeń')
plt.ylabel('Średni wskaźnik anulacji')
plt.legend(title='Segment klienta', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

Sezonowość segmentów

In [ ]:
target = 'booking_status'

df_plot = df.copy()

month_segment_cancel = (
    df_plot.groupby(['arrival_month', 'market_segment_type'])[target]
    .mean()
    .reset_index()
)

month_segment_count = (
    df_plot.groupby(['arrival_month', 'market_segment_type'])
    .size()
    .reset_index(name='n')
)

month_segment_summary = month_segment_cancel.merge(
    month_segment_count, on=['arrival_month', 'market_segment_type']
)

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=month_segment_summary,
    x='arrival_month',
    y=target,
    hue='market_segment_type',
    marker='o'
)

plt.title('Wskaźnik anulacji wg miesiąca przyjazdu (wg segmentu klienta)')
plt.xlabel('Miesiąc przyjazdu')
plt.ylabel('Średni wskaźnik anulacji')
plt.xticks(range(1, 13))
plt.legend(title='Segment klienta', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### **Wnioski biznesowe**

- Warto wdrożyć różne strategie dla rezerwacji z dużym wyprzedzeniem, np. przypomnienia o rezerwacjach, częściowe wpłaty zaliczek.
- Dla droższych rezerwacji warto stosować działania ogtaniczające anulacje, np. benefity 
- Warto aktywnie zachęcać klientów do podawania preferencji po rezerwacji, gdyż może to zmniejszyć ryzyko anulowania rezerwacji
- należy stosować inaczej zarządzać rezewacjami między konkretnymi segmentami, gdyż znacznie różnią się między sobą
- najbardziej ryzykowny profil rezerwacji to offline oraz online
- należy szczególnie zadbać o ograniczenie anulowania rezerwacji w miesiącach letnich/ wiosennych dla segmentu Coroprate